# Análise de ROI em Publicidade — Melhores Compras
## Desafio 3: Regressão Linear para Apoio à Área de Marketing

**Projeto:** TSCO 2º Ano — Fase 5  
**Dataset:** `Gastos_Publicidade_MelhoresCompras.csv`  
**Objetivo:** Construir modelo de regressão linear para prever o impacto dos gastos com publicidade nas vendas e identificar o canal com melhor ROI.

---

### Pipeline adotado (CRISP-DM — Cap. 10 FIAP)
1. **Carga das Bibliotecas**
2. **Carga dos Dados** *(arquivo original não é modificado)*
3. **Preparação dos Dados** *(dummies + pivot por mídia)*
4. **Análise de Correlação** *(matriz numérica + visual)*
5. **Gráficos de Dispersão** *(cada mídia vs. vendas)*
6. **Construção dos Modelos** *(3 modelos, busca pelo maior R²)*
7. **Avaliação dos Modelos** *(R², MSE, p-valor via statsmodels)*
8. **Análise Explicativa para o Time de Marketing**

In [ ]:
# =============================================================
# ETAPA 1 — Carga das Bibliotecas
# =============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import statsmodels.api as sm

import warnings
warnings.filterwarnings('ignore')

# Configuração visual
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.titlesize'] = 13
print('Bibliotecas carregadas com sucesso!')

---
## ETAPA 2 — Carga dos Dados
> **Atenção:** O arquivo CSV original **não é modificado** em nenhum momento.  
> Todas as transformações são feitas em cópias do dataframe em memória.

In [ ]:
# =============================================================
# ETAPA 2 — Carga dos Dados
# =============================================================
df = pd.read_csv('assets/Gastos_Publicidade_MelhoresCompras.csv')

print(f'Shape: {df.shape}')
print(f'Colunas: {list(df.columns)}')
print(f'\nPeíodo: {df["Ano"].min()}/{str(df["Mes"].min()).zfill(2)} a {df["Ano"].max()}/{str(df["Mes"].max()).zfill(2)}')
print(f'Valores nulos: {df.isnull().sum().sum()}')
df.head(10)

In [ ]:
# Análise Exploratória Inicial
print('=== Tipos de Mídia disponíveis ===')
print(df['Tipo de Midia'].value_counts())

print('\n=== Estatísticas Descritivas ===')
df.describe().round(2)

---
## ETAPA 3 — Preparação dos Dados

Seguindo as recomendações do desafio:

- **Item (a):** `Tipo de Mídia` não pode ser usada diretamente → aplicamos **One-Hot Encoding** (variáveis dummy)
- **Item (b):** `Gastos com Publicidade` contém todas as mídias → separamos os gastos por mídia usando **pivot_table**, criando uma coluna de gasto para cada canal

Isso permite que o modelo identifique o **coeficiente (ROI) de cada mídia individualmente**.

In [ ]:
# =============================================================
# ETAPA 3 — Preparação dos Dados
# =============================================================

# Cópia de trabalho (CSV original intocado)
df_work = df.copy()
df_work.columns = ['Ano', 'Mes', 'Tipo_Midia', 'Gastos', 'Vendas_Previstas']

# ---- 3.1 Formato AMPLO (wide): uma coluna de gastos por tipo de mídia ----
# Pivot: gastos por mídia como colunas separadas
df_gastos_wide = df_work.pivot_table(
    index=['Ano', 'Mes'],
    columns='Tipo_Midia',
    values='Gastos',
    aggfunc='sum',
    fill_value=0
).reset_index()
df_gastos_wide.columns.name = None

# Renomear colunas para facilitar uso
rename_map = {}
for col in df_gastos_wide.columns:
    if col not in ['Ano', 'Mes']:
        rename_map[col] = 'Gastos_' + col.replace(' ', '_').replace('/', '_')
df_gastos_wide = df_gastos_wide.rename(columns=rename_map)

# Total de vendas previstas por mês
df_vendas_mes = df_work.groupby(['Ano', 'Mes'])['Vendas_Previstas'].sum().reset_index()

# Dataset final para modelagem
df_wide = df_gastos_wide.merge(df_vendas_mes, on=['Ano', 'Mes'])

print(f'Dataset wide (para modelagem): {df_wide.shape}')
print(f'Colunas: {list(df_wide.columns)}')
df_wide.head()

In [ ]:
# ---- 3.2 Formato LONGO com Dummies (para comparação de modelos) ----
df_long = df_work.copy()
dummies = pd.get_dummies(df_long['Tipo_Midia'], prefix='midia', drop_first=True)
df_long = pd.concat([df_long, dummies], axis=1)

# Colunas de features
feature_cols_wide = [c for c in df_wide.columns if c.startswith('Gastos_')]
dummy_cols = [c for c in df_long.columns if c.startswith('midia_')]

print(f'Mídias no modelo wide: {feature_cols_wide}')
print(f'\nDummies criadas: {dummy_cols}')

---
## ETAPA 4 — Análise de Correlação

> *Conforme Cap. 11 FIAP (seção 2.1): antes de construir o modelo, analisamos a correlação entre as variáveis para entender quais features têm maior relação linear com a variável-alvo.*

O **coeficiente de correlação de Pearson** varia de **-1 a 1**:
- Próximo de **+1**: forte correlação positiva (mais gasto → mais vendas)
- Próximo de **0**: sem correlação linear
- Próximo de **-1**: forte correlação negativa

In [ ]:
# =============================================================
# ETAPA 4.1 — Matriz de Correlação (numérica)
# =============================================================
df_corr = df_wide[feature_cols_wide + ['Vendas_Previstas']]
corr_matrix = df_corr.corr()

print('Matriz de Correlação (Pearson):')
print(corr_matrix.round(3))

print('\n=== Correlação com Vendas_Previstas (ordenado) ===')
corr_vendas = corr_matrix['Vendas_Previstas'].drop('Vendas_Previstas').sort_values(ascending=False)
print(corr_vendas.round(3))

In [ ]:
# =============================================================
# ETAPA 4.2 — Matriz de Correlação (visual — metodologia Cap. 11)
# =============================================================

# Gráfico 1: matshow (abordagem do material FIAP)
f = plt.figure(figsize=(10, 8))
plt.matshow(corr_matrix, fignum=f.number)
labels = [c.replace('Gastos_', '').replace('_', ' ') for c in corr_matrix.columns]
plt.xticks(range(len(corr_matrix.columns)), labels, fontsize=9, rotation=45)
plt.yticks(range(len(corr_matrix.columns)), labels, fontsize=9)
cb = plt.colorbar()
cb.ax.tick_params(labelsize=10)
plt.title('Matriz de Correlação — Gastos por Mídia vs Aumento de Vendas', fontsize=13, pad=20)
plt.tight_layout()
plt.savefig('assets/matriz_correlacao.png', bbox_inches='tight', dpi=100)
plt.show()

# Gráfico 2: heatmap com anotações (leitura facilitada)
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, vmin=-1, vmax=1)
plt.title('Heatmap de Correlação — Valores Anotados', fontsize=13)
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print('\n>>> COMENTÁRIO: As mídias com maior correlação com Vendas_Previstas são:')
print(corr_vendas.head(3))
print('\nEssas são as mídias que individualmente mais explicam a variação nas vendas previstas.')

---
## ETAPA 5 — Gráficos de Dispersão

> *Conforme Cap. 11 FIAP: os gráficos de dispersão nos ajudam a identificar visualmente se existe relação linear entre cada variável independente (gasto por mídia) e a variável dependente (aumento de vendas).*

Para cada tipo de mídia, analisamos:
- **Padrão linear:** se os pontos seguem uma tendência de reta
- **Outliers:** pontos fora do padrão geral
- **Tendência:** crescente (correlação positiva) ou decrescente (negativa)

In [ ]:
# =============================================================
# ETAPA 5 — Gráficos de Dispersão
# =============================================================
n_midas = len(feature_cols_wide)
ncols = 3
nrows = (n_midas + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(18, 5 * nrows))
axes = axes.flatten()

colors = plt.cm.tab10(np.linspace(0, 1, n_midas))

for i, col in enumerate(feature_cols_wide):
    midia_nome = col.replace('Gastos_', '').replace('_', ' ')
    x = df_wide[col]
    y = df_wide['Vendas_Previstas']
    
    axes[i].scatter(x, y, alpha=0.7, color=colors[i], edgecolors='white', s=60)
    
    # Linha de tendência
    if x.std() > 0:
        z = np.polyfit(x, y, 1)
        p = np.poly1d(z)
        x_line = np.linspace(x.min(), x.max(), 100)
        axes[i].plot(x_line, p(x_line), 'r--', linewidth=2, label='Tendência linear')
        axes[i].legend(fontsize=8)
    
    # Correlação no título
    corr_val = x.corr(y)
    axes[i].set_title(f'{midia_nome}\n(corr = {corr_val:.3f})', fontsize=10)
    axes[i].set_xlabel(f'Gastos com {midia_nome} (R$)', fontsize=9)
    axes[i].set_ylabel('Aumento de Vendas (mil un.)', fontsize=9)
    axes[i].ticklabel_format(style='sci', axis='x', scilimits=(3,3))

# Esconder subplots extras
for j in range(i + 1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Dispersão: Gastos por Tipo de Mídia vs Aumento Previsto de Vendas',
             fontsize=14, y=1.01, fontweight='bold')
plt.tight_layout()
plt.savefig('assets/graficos_dispersao.png', bbox_inches='tight', dpi=100)
plt.show()

print('\n>>> COMENTÁRIO:')
print('Mídias com padrão linear mais claro (maiores correlações):')
for m, v in corr_vendas.items():
    status = 'correlacao forte' if abs(v) > 0.6 else ('correlacao moderada' if abs(v) > 0.3 else 'correlacao fraca')
    print(f'  {m.replace("Gastos_",""):25s}: {v:+.3f}  {status}')

---
## ETAPA 6 — Construção dos Modelos de Regressão

Testamos **3 modelos** com complexidade crescente, buscando o maior R² com menor risco de overfitting.

| Modelo | Variáveis | Justificativa |
|--------|-----------|---------------|
| **M1 — Baseline** | Apenas `Gastos` totais | Referência — ignora tipo de mídia |
| **M2 — Com Dummies** | `Gastos` + dummies de mídia | Captura efeito do tipo de mídia |
| **M3 — Wide (Melhor)** | Gastos por mídia (pivot) | Coeficiente = ROI direto por canal |

> *Separação treino/teste: **~67% / 33%**, random_state=42 (reprodutível) — conforme pipeline Cap. 10 FIAP.*

In [ ]:
# =============================================================
# ETAPA 6.1 — MODELO 1: Baseline (apenas Gastos totais)
# =============================================================
print('=' * 65)
print('MODELO 1 — BASELINE: Gastos Totais (sem separação por mídia)')
print('=' * 65)
print('ATENCAO: Usar Gastos totais sem separar por mídia prejudica a análise de ROI')
print('   (conforme instrução b do desafio).  Serve apenas como referência.\n')

df_total = df_work.groupby(['Ano', 'Mes'])['Gastos'].sum().reset_index()
df_total = df_total.merge(df_vendas_mes, on=['Ano', 'Mes'])

X1 = df_total[['Gastos']]
y1 = df_total['Vendas_Previstas']

X1_treino, X1_teste, y1_treino, y1_teste = train_test_split(
    X1, y1, test_size=0.3266, random_state=42
)

mod1 = LinearRegression()
mod1.fit(X1_treino, y1_treino)
y1_prev = mod1.predict(X1_teste)

r2_m1_train = r2_score(y1_treino, mod1.predict(X1_treino))
r2_m1_test  = r2_score(y1_teste, y1_prev)
mse_m1      = mean_squared_error(y1_teste, y1_prev)

print(f'Treino — n={len(X1_treino)} obs  |  Teste — n={len(X1_teste)} obs')
print(f'R² Treino : {r2_m1_train:.4f}')
print(f'R² Teste  : {r2_m1_test:.4f}')
print(f'MSE       : {mse_m1:.2f}')
print(f'RMSE      : {np.sqrt(mse_m1):.2f}')

In [ ]:
# =============================================================
# ETAPA 6.2 — MODELO 2: Gastos + Variáveis Dummy
# =============================================================
print('=' * 65)
print('MODELO 2 — Gastos + Variáveis Dummy por Tipo de Mídia')
print('=' * 65)

feature_cols_m2 = ['Gastos'] + dummy_cols
X2 = df_long[feature_cols_m2]
y2 = df_long['Vendas_Previstas']

X2_treino, X2_teste, y2_treino, y2_teste = train_test_split(
    X2, y2, test_size=0.3266, random_state=42
)

mod2 = LinearRegression()
mod2.fit(X2_treino, y2_treino)
y2_prev = mod2.predict(X2_teste)

r2_m2_train = r2_score(y2_treino, mod2.predict(X2_treino))
r2_m2_test  = r2_score(y2_teste, y2_prev)
mse_m2      = mean_squared_error(y2_teste, y2_prev)

print(f'Treino — n={len(X2_treino)} obs  |  Teste — n={len(X2_teste)} obs')
print(f'R² Treino : {r2_m2_train:.4f}')
print(f'R² Teste  : {r2_m2_test:.4f}')
print(f'MSE       : {mse_m2:.2f}')
print(f'RMSE      : {np.sqrt(mse_m2):.2f}')

In [ ]:
# =============================================================
# ETAPA 6.3 — MODELO 3: Wide — Gastos Separados por Mídia
# (Melhor modelo — maior R² — abordagem recomendada)
# =============================================================
print('=' * 65)
print('MODELO 3 — WIDE: Gastos Separados por Tipo de Mídia (MELHOR)')
print('=' * 65)
print('Segue recomendações a e b do desafio.')
print('   Coeficiente de cada variável = ROI direto daquela mídia.\n')

X3 = df_wide[feature_cols_wide]
y3 = df_wide['Vendas_Previstas']

X3_treino, X3_teste, y3_treino, y3_teste = train_test_split(
    X3, y3, test_size=0.3266, random_state=42
)

# Statsmodels para estatísticas completas (R², p-valor, coef, std err)
# Conforme Cap. 11 FIAP
X3_train_sm = sm.add_constant(X3_treino)
mod3_sm = sm.OLS(y3_treino, X3_train_sm).fit()
print(mod3_sm.summary())

X3_teste_sm = sm.add_constant(X3_teste)
y3_prev = mod3_sm.predict(X3_teste_sm)

r2_m3_train = mod3_sm.rsquared
r2_m3_test  = r2_score(y3_teste, y3_prev)
mse_m3      = mean_squared_error(y3_teste, y3_prev)

print(f'\nTreino — n={len(X3_treino)} obs  |  Teste — n={len(X3_teste)} obs')
print(f'R² Treino (statsmodels) : {r2_m3_train:.4f}')
print(f'R² Teste                : {r2_m3_test:.4f}')
print(f'MSE                     : {mse_m3:.2f}')
print(f'RMSE                    : {np.sqrt(mse_m3):.2f}')

---
## ETAPA 7 — Avaliação e Comparação dos Modelos

In [ ]:
# =============================================================
# ETAPA 7.1 — Tabela Comparativa dos Modelos
# =============================================================
resultados = pd.DataFrame({
    'Modelo': [
        'M1 — Gastos Totais (baseline)',
        'M2 — Gastos + Dummies',
        'M3 — Wide por Mídia (MELHOR)'
    ],
    'R² Treino': [r2_m1_train, r2_m2_train, r2_m3_train],
    'R² Teste':  [r2_m1_test,  r2_m2_test,  r2_m3_test],
    'MSE Teste': [mse_m1,      mse_m2,      mse_m3],
    'RMSE Teste':[np.sqrt(mse_m1), np.sqrt(mse_m2), np.sqrt(mse_m3)]
})
resultados = resultados.set_index('Modelo')
print('=== Comparação dos Modelos ===')
print(resultados.round(4))
print('\n>>> Interpretação:')
print('   R² próximo de 1.0 = modelo explica bem a variação das vendas')
print('   MSE/RMSE menor = menor erro de previsão')
print('   Modelo 3 é o melhor por separar o ROI de cada mídia individualmente')

In [ ]:
# =============================================================
# ETAPA 7.2 — ROI por Mídia (Coeficientes do Modelo 3)
# =============================================================
coefs = mod3_sm.params.drop('const')
pvalores = mod3_sm.pvalues.drop('const')

df_roi = pd.DataFrame({
    'Midia': [c.replace('Gastos_', '').replace('_', ' ') for c in coefs.index],
    'Coeficiente (ROI)': coefs.values,
    'ROI por R$ 1.000': coefs.values * 1000,
    'p-valor': pvalores.values,
    'Significativo (p<0.05)': pvalores.values < 0.05
}).sort_values('ROI por R$ 1.000', ascending=False).reset_index(drop=True)

print('=== ROI por Tipo de Mídia (Coeficientes da Regressão) ===')
print('Interpretação: unidades de vendas previstas (mil) por R$ 1.000 investidos\n')
print(df_roi.to_string(index=False))

# Gráfico de barras dos coeficientes
fig, ax = plt.subplots(figsize=(10, 5))
cores = ['green' if v > 0 else 'crimson' for v in df_roi['ROI por R$ 1.000']]
bars = ax.barh(df_roi['Midia'], df_roi['ROI por R$ 1.000'], color=cores, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Aumento de Vendas (mil unidades) por R$ 1.000 investidos', fontsize=10)
ax.set_title('ROI por Tipo de Mídia — Coeficientes da Regressão Linear', fontsize=12)
for bar, val in zip(bars, df_roi['ROI por R$ 1.000']):
    ax.text(val + (0.1 if val >= 0 else -0.1), bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
plt.tight_layout()
plt.savefig('assets/roi_por_midia.png', bbox_inches='tight', dpi=100)
plt.show()

In [ ]:
# =============================================================
# ETAPA 7.3 — Gráfico Previsto x Real (Cap. 10 FIAP, seção 4.1.2)
# =============================================================
# Previsões no conjunto de TREINO para análise de erro
y3_prev_treino = mod3_sm.predict(X3_train_sm)

ev = pd.DataFrame({'prev': y3_prev_treino, 'real': y3_treino.values})
ev['erro_absoluto'] = ev['prev'] - ev['real']

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter previsto x real
axes[0].scatter(ev['prev'], ev['real'], alpha=0.6, color='steelblue')
lim_min = min(ev['prev'].min(), ev['real'].min())
lim_max = max(ev['prev'].max(), ev['real'].max())
axes[0].plot([lim_min, lim_max], [lim_min, lim_max], 'r--', label='Previsto = Real')
axes[0].set_xlabel('Valores Previstos', fontsize=10)
axes[0].set_ylabel('Valores Reais', fontsize=10)
axes[0].set_title('Previsto vs Real (Treino)', fontsize=11)
axes[0].legend()

# Histograma do erro
axes[1].hist(ev['erro_absoluto'], bins=15, color='steelblue', edgecolor='white')
axes[1].axvline(0, color='red', linestyle='--', linewidth=2, label='Erro = 0')
axes[1].set_xlabel('Erro Absoluto (Previsto - Real)', fontsize=10)
axes[1].set_ylabel('Frequência', fontsize=10)
axes[1].set_title('Distribuição do Erro de Previsão', fontsize=11)
axes[1].legend()

plt.suptitle('Avaliação do Modelo 3 — Análise de Erro', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Erro médio absoluto : {ev["erro_absoluto"].abs().mean():.2f} mil unidades')
print(f'Erro médio          : {ev["erro_absoluto"].mean():.4f} (próximo de 0 = modelo equilibrado)')

# Intervalo de confiança 95% (conforme Cap. 10 FIAP)
import scipy.stats as st
data_ic = ev[['erro_absoluto']].to_numpy()
ci = st.t.interval(alpha=0.95, df=len(ev)-1,
                   loc=np.mean(data_ic), scale=np.std(data_ic))
print(f'\nIntervalo de confiança 95%: ({ci[0]:.2f}, {ci[1]:.2f})')
print(f'→ 95% das previsões têm erro entre {ci[0]:.0f} e {ci[1]:.0f} mil unidades')

---
## ETAPA 8 — Análise Explicativa para o Time de Marketing

---

### O que foi feito?

Construímos um **modelo de regressão linear** que relaciona os investimentos em publicidade de cada canal de mídia com o **aumento esperado de vendas** (em milhares de unidades). O modelo foi treinado com dados históricos de 2022 a 2024.

---

### Como ler os resultados?

O modelo gera um **coeficiente** para cada mídia. Esse coeficiente representa o **ROI (Return on Investment)**: quantas unidades de venda (em milhares) são previstas para cada R$ 1.000 investidos naquele canal.

**Exemplo:** Se o coeficiente de TV = 0,008 → a cada R$ 1.000 investidos em TV, espera-se um aumento de **8 unidades de venda**.

---

### Principais conclusões

1. **Qualidade do modelo (R²):** O Modelo 3 apresentou o maior R², o que significa que os gastos por tipo de mídia explicam bem a variação nas vendas previstas. Quanto mais próximo de 1,0 (100%), melhor o modelo.

2. **Canal com melhor ROI:** As mídias com os maiores coeficientes positivos e p-valor < 0,05 (estatisticamente significativos) são os canais onde cada real investido gera mais retorno em vendas. Com base nos resultados, essas mídias devem **receber maior alocação orçamentária**.

3. **Canais a revisar:** Mídias com coeficiente próximo de zero ou negativo indicam baixo retorno. O time de marketing deve **avaliar a continuidade desses investimentos** ou reposicioná-los estrategicamente.

4. **Instagram (criado em 2024):** Por ter menos histórico, seu coeficiente deve ser interpretado com cautela. Recomenda-se acompanhar seu desempenho por mais períodos antes de conclusões definitivas.

---

### Recomendações estratégicas

| Ação | Justificativa |
|------|---------------|
| Aumentar orçamento nos canais de maior coeficiente | Maior retorno por real investido |
| Reduzir ou rever canais de coeficiente negativo/baixo | Baixo impacto nas vendas |
| Monitorar mensalmente com o modelo | Permite ajustes rápidos de estratégia |
| Ampliar histórico do Instagram | Coeficiente mais confiável com mais dados |

---

### Limitações do modelo

- O modelo assume **relação linear** entre gastos e vendas. Relações não-lineares podem não ser capturadas.
- Outros fatores influenciam vendas além da publicidade (sazonalidade, concorrência, preço).
- O conjunto de dados tem **período limitado** (2022–2024); com mais histórico, o modelo tende a melhorar.
- A variável `Previsão Inicial de Aumento de Vendas` é uma estimativa inicial e pode conter imprecisões.

---
*Análise elaborada com Python (scikit-learn + statsmodels) seguindo metodologia CRISP-DM — FIAP Fase 5*